# ASD Scanpath Classifier Training — all models, all configs

One notebook to train every model explored for the ASD eye-tracking paper:
- **12 PyTorch configs**: 2 backbones (MobileNetV4-small, ConvNeXtV2-tiny) x 3
  GhostNet insertion variants (PA1/PA2/PA3) x with/without CBAM attention.
- **1 TensorFlow config**: the accuracy-focused InceptionV3 track (97% reported
  accuracy in the paper).

All model/training code lives in `src/asd_train/` — this notebook only picks
configs and calls into it. See `README.md` for what each config means and
its reported accuracy.

**Framework note:** TensorFlow and PyTorch both try to claim the GPU. Run the
PyTorch section OR the TensorFlow section in a given kernel session, not both
back to back without a restart.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# Make src/ importable (works whether this notebook runs from a local git
# clone or from the Drive-mirrored code/ folder in Colab).
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET_ROOT = "/content/drive/MyDrive/ASD/ML4Autism/train/datasets"
except ImportError:
    # Local run: dataset lives in the meta-repo's data/ (sibling of this
    # train/ repo, NOT inside it) -- see ASD/.agents/record.md (T4) for how
    # it's populated (a local copy, not the slow Drive FUSE mount).
    DATASET_ROOT = str((REPO_ROOT / ".." / "data").resolve())

OUTPUT_DIR = str(REPO_ROOT / "artifacts")
print(f"DATASET_ROOT = {DATASET_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")

## PyTorch — Ghost / Ghost+CBAM (12 configs)

`PRESETS` has one `GhostRunConfig` per config, with every hyperparameter
read directly from the original notebooks (see `src/asd_train/configs.py`
docstring).

In [ ]:
from asd_train.configs import PRESETS, BEST_PRESETS

print(f"{len(PRESETS)} configs available:")
for name in PRESETS:
    print(" -", name)

### Run a single config

In [ ]:
from asd_train.train_torch import run_torch_experiment

config_name = "ghost_mobilenetv4_pa2"  # <- change to any key in PRESETS
result = run_torch_experiment(PRESETS[config_name], dataset_root=DATASET_ROOT, output_dir=OUTPUT_DIR)
result

### Run all 12 configs and collect a results table

In [ ]:
results = {}
for name, cfg in PRESETS.items():
    print(f"\n{'='*80}\n{name}\n{'='*80}")
    results[name] = run_torch_experiment(cfg, dataset_root=DATASET_ROOT, output_dir=OUTPUT_DIR)

for name, r in results.items():
    print(f"{name:35s} acc={r.test_accuracy:.4f}  precision={r.test_precision:.4f}  recall={r.test_recall:.4f}")

## TensorFlow — InceptionV3 (97% reported accuracy)

**Restart the kernel before running this section** if you already ran the
PyTorch cells above in this session.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from asd_train.configs import INCEPTIONV3_CONFIG
from asd_train.train_tf import run_inceptionv3_experiment

DATASET_ROOT = str((Path.cwd() / ".." / "data").resolve())  # re-set if kernel was restarted
OUTPUT_DIR = "artifacts"

tf_result = run_inceptionv3_experiment(INCEPTIONV3_CONFIG, dataset_root=DATASET_ROOT, output_dir=OUTPUT_DIR)
tf_result